# Artpedia Image Captioning — Full Pipeline (LOCAL)

> **Local version** — runs on your own machine inside the `lavis` conda environment.
> The Colab version is `01_image_to_text_finetuning.ipynb`.

Pipeline:
1. **Data prep** — download and cache Artpedia images to local Drive
2. **Fine-tuning** — partially fine-tune BLIP's text decoder on the train split
3. **Evaluation** — BLEU-4 and CIDEr for pretrained vs fine-tuned
4. **t-SNE** — visualise how fine-tuning shifts decoder representations
5. **VizWiz** — zero-shot robustness evaluation on real-world images
6. **MLflow** — notes on viewing logged metrics

## Prerequisites

Assumes the `lavis` conda environment with LAVIS and all dependencies already installed.
Run this notebook **from the notebooks folder** (`mscai-multimodal-project/notebooks`).

```
conda activate lavis
jupyter notebook
```

In [ ]:
# Local paths
DATA_DIR    = "../data"
OUTPUTS_DIR = "../outputs"   # all run artefacts (checkpoints, eval, t-SNE, MLflow); gitignored
CHECKPOINT  = None           # populated by the auto-detect cell after fine-tuning

print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUTS_DIR = {OUTPUTS_DIR}")
print(f"CHECKPOINT  = {CHECKPOINT}")

## 1. Data Preparation

Downloads Artpedia images from Wikimedia (500 px thumbnails) and writes manifests.
Images already on disk are reused — safe to rerun after interruptions.

Run the **test or validation split first** (326 images, good for verifying setup).
The train split cell is commented out — uncomment and run deliberately once test is confirmed working.
Add `--rebuild-manifest` to regenerate the manifest while reusing cached images.

In [ ]:
# Download and cache the TEST split (339 records).
!python ../src/prepare_data.py \
    --split test \
    --output-dir "{DATA_DIR}" \
    --delay 2.0

In [ ]:
# Download and cache the Validation split (339 records).
!python ../src/prepare_data.py \
    --split val \
    --output-dir "{DATA_DIR}" \
    --delay 2.0

In [ ]:
# Download and cache the TRAIN split (2252 records).

# !python ..src/prepare_data.py \
#     --split train \
#     --output-dir "{DATA_DIR}" \
#     --delay 2.0

# To regenerate the manifest from existing images (e.g. after a path format change):
# !python src/prepare_data.py --split train --output-dir "{DATA_DIR}" --rebuild-manifest

## 2. Fine-tuning

Fine-tunes BLIP's **text decoder** (last 4 transformer layers + cls head) while keeping the **vision encoder frozen**.

Checkpoints are saved to `CKPT_DIR` after each epoch.
Early stopping is active (patience=10) — training halts when loss stops improving.
If you hit a CUDA OOM error, reduce `--batch-size`.

In [ ]:
!python ../src/train_caption.py \
    --manifest        "{DATA_DIR}/processed/train.jsonl" \
    --base-dir        "{DATA_DIR}" \
    --output-dir      "{OUTPUTS_DIR}" \
    --epochs          10 \
    --batch-size      8 \
    --lr              1e-5 \
    --unfreeze-last-n 2 \
    --fp16 \
    --num-workers     0 \
    --use-mlflow \
    --mlflow-experiment blip-artpedia

In [ ]:
# Auto-detect the latest training run and pick the best checkpoint.
# Set EPOCH to a specific integer (e.g. 5) to pick that epoch's checkpoint instead.
import glob, os

EPOCH = None   # None → blip_artpedia_best.pth;  integer → blip_artpedia_epoch<N>.pth

run_dirs = sorted(glob.glob(OUTPUTS_DIR + "/train_*"))
if not run_dirs:
    print("No training runs found in", OUTPUTS_DIR, "— run the training cell first.")
else:
    latest_run = run_dirs[-1]
    if EPOCH is not None:
        candidate  = os.path.join(latest_run, f"blip_artpedia_epoch{EPOCH}.pth")
        CHECKPOINT = candidate if os.path.exists(candidate) \
                     else os.path.join(latest_run, "blip_artpedia_best.pth")
    else:
        CHECKPOINT = os.path.join(latest_run, "blip_artpedia_best.pth")
    print(f"CHECKPOINT = {CHECKPOINT}")
    print(f"Exists     : {os.path.exists(CHECKPOINT)}")

## 3. Evaluation

Computes **BLEU-4** and **CIDEr** on the test split for both the pretrained base model and the fine-tuned checkpoint.
Results are saved to `eval_results.json` and logged to MLflow.

Update `CHECKPOINT` in the paths cell above to point to the epoch you want to evaluate.

In [ ]:
!python ../src/evaluate_caption.py \
    --manifest      "{DATA_DIR}/processed/test.jsonl" \
    --base-dir      "{DATA_DIR}" \
    --checkpoint    "{CHECKPOINT}" \
    --split         test \
    --output-dir    "{OUTPUTS_DIR}" \
    --use-mlflow \
    --mlflow-experiment blip-artpedia \
    --run-name      local-eval

## 4. t-SNE Comparison

Visualises how fine-tuning shifts the **decoder's multimodal representations** (cross-attention hidden states, not raw image embeddings) on the test set, coloured by painting century.

In [ ]:
!python ../src/tsne_compare.py \
    --manifest   "{DATA_DIR}/processed/test.jsonl" \
    --base-dir   "{DATA_DIR}" \
    --checkpoint "{CHECKPOINT}" \
    --split      test \
    --output-dir "{OUTPUTS_DIR}" \
    --output     tsne_compare.png

In [ ]:
# Display the most recent t-SNE plot inline.
import glob
from IPython.display import Image as IPImage

tsne_dirs = sorted(glob.glob(OUTPUTS_DIR + "/tsne_*"))
if tsne_dirs:
    pngs = sorted(glob.glob(tsne_dirs[-1] + "/*.png"))
    if pngs:
        print(f"Displaying: {pngs[0]}")
        display(IPImage(pngs[0]))
    else:
        print("No PNG found in", tsne_dirs[-1])
else:
    print("No t-SNE runs found in", OUTPUTS_DIR, "— run the t-SNE cell first.")

## 5. MLflow — Viewing Results

MLflow writes logs to `../outputs/mlruns/` (all scripts share the same root so runs from training, evaluation, and VizWiz appear in the same UI).

**To view the UI**, from the repo root run:
```
mlflow ui --backend-store-uri outputs/mlruns
```
then open `http://127.0.0.1:5000` in your browser.

## 6. Zero-shot Robustness Evaluation on VizWiz

[VizWiz-Caps](https://vizwiz.org/tasks-and-datasets/image-captioning/) contains photos taken by blind people —
a real-world domain-robustness test for the museum-adapted model.
Images stream from HuggingFace (`lmms-lab/VizWiz-Caps`); no full dataset download needed.
Adjust `--limit` to control how many samples are evaluated.

In [ ]:
# CHECKPOINT (set by the auto-detect cell) points to the fine-tuned model.
# Adjust --limit to control how many VizWiz samples are evaluated.
# Add --hf-timeout <seconds> (default 120) if you hit ReadTimeoutError on a slow connection.
!python ../src/evaluate_vizwiz.py \
    --checkpoint "{CHECKPOINT}" \
    --limit      1 \
    --output-dir "{OUTPUTS_DIR}" \
    --use-mlflow \
    --run-name   vizwiz-zeroshot